In [ ]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.insert(0,"..")
from src.vocab.filtering import is_plausible_number_token, is_plausible_string_token, is_plausible_boolean_token, build_plausible_vocab, is_plausible_token_name
from llm_sdk import Small_LLM_Model
from src.vocab.loader import loader_vocab
from src.fsm.value_matcher import generate_boolean_value, generate_number_value
from src.generation.decoding import decode_vocab
from src.fsm.value_matcher import generate_string_value
from src.fsm.name_matcher import name_matcher
from src.models.load_init import load_function_catalog, what_function_are_calling, build_function_definition
from src.models.load_init import FunctionCatalog
from src.generation.orchestrator import generate_function_call

In [ ]:
catalog = load_function_catalog("../data/input/functions_definition.json")
descriptions = build_function_definition(catalog)
initial_context = f"Available functions:\n{descriptions}\n\n"

In [ ]:
prompts = "Replace all numbers in \"Hello 34 I'm 233 years old\" with NUMBERS"
full_text_json = f'{{"prompt": "{prompts}", "name": "'
full_text_context = initial_context + full_text_json
model = Small_LLM_Model()
dico_inverse = loader_vocab(model)
dico_decode = decode_vocab(dico_inverse, model)
allowed_characters = {c for w in catalog.functions for c in w.name} 
plausible_number = build_plausible_vocab(dico_inverse, is_plausible_number_token)
plausible_string = build_plausible_vocab(dico_decode, is_plausible_string_token)
plausible_boolean = build_plausible_vocab(dico_inverse, is_plausible_boolean_token)
plausible_token_name = build_plausible_vocab(dico_inverse, lambda x: is_plausible_token_name(x, allowed_characters))
noms_valide = [w.name for w in catalog.functions]


In [ ]:
prompt_liste = [
    "What is the sum of 2 and 3?",
    "What is the sum of 265 and 345?",
    "Greet shrek",
    "Greet john",
    "Reverse the string 'hello'",
    "Reverse the string 'world'",
    "What is the square root of 16?",
    "Calculate the square root of 144",
    "Replace all numbers in \"Hello 34 I'm 233 years old\" with NUMBERS",
    "Replace all vowels in 'Programming is fun' with asterisks",
    "Substitute the word 'cat' with 'dog' in 'The cat sat on the mat with another cat'"
]
for prompt in prompt_liste:
    result = generate_function_call(
        prompt,
        catalog,
        model,
        initial_context,
        noms_valide,
        plausible_token_name,
        plausible_number,
        plausible_string,
        plausible_boolean,
    )
    print(result)